In [ ]:
import tensorflow as tf 
from tensorflow.keras.applications import ResNet50 
from tensorflow.keras.layers import Dense, Flatten, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
# Memuat model dasar ResNet50 (Transfer Learning)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Membekukan semua lapisan dasar agar tidak berubah
for layer in base_model.layers:
    layer.trainable = False


In [ ]:
# Menambahkan 'Kepala' Baru untuk 2 kelas (Kucing dan Anjing)
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dense(2, activation='softmax')(x)  # 2 Kelas (Kucing, Anjing)

# Menyatukan model
model = Model(inputs=base_model.input, outputs=x)

# Compile model awal
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# model.summary() # Hilangkan komentar jika ingin melihat arsitektur


In [ ]:
# --- PERSIAPAN DATA ---
# Pastikan Anda punya folder gambar! 
# Ganti tulisan 'NAMA_FOLDER_DATASET_ANDA' dengan lokasi folder asli di komputer Anda.
dataset_path = r'd:\AI_Engeener\DeepLearn\dataset'

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = datagen.flow_from_directory(
    dataset_path, 
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    dataset_path, 
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)


In [ ]:
# Membuka 10 lapisan terakhir untuk Fine-Tuning
for layer in base_model.layers[-10:]:
    layer.trainable = True

# Compile ulang dengan learning rate sangat kecil
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), 
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Mulai Melatih AI (Hilangkan tanda komentar # di bawah ini jika dataset_path sudah benar!)
history = model.fit(train_data, epochs=5, validation_data=val_data)
